In [22]:
import torch
import numpy as np
import pandas as pd
from poincare import PoincareManifold
from sklearn.metrics import pairwise_distances
from sklearn.metrics import average_precision_score
from data import prepare_data

In [1]:
import pandas as pd
profils_omim = pd.read_csv("../data/profils_omim.csv.gz", index_col=0)
profils_omim = profils_omim.reset_index()
#print("")

#print("="*10, "Préparation des données", "="*10)
#x, features, labels = prepare_data(profils_omim, with_labels=True, normalize=False, n_pca=N_PCA)

/tmp/ipykernel_9391/2316400135.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  profils_omim = profils_omim.reset_index()


In [21]:
duplicates = profils_omim[profils_omim.drop(columns='gene_maladie_assoc').duplicated(keep=False)]
print(f"Nombre de lignes dupliquées : {len(duplicates)}")

Nombre de lignes dupliquées : 265


,gene_maladie_assoc,HP:0000002,HP:0000003,HP:0000008,HP:0000009,HP:0000010,HP:0000011,HP:0000012,HP:0000013,HP:0000014,...,HP:6000590,HP:6000768,HP:6000818,HP:6000852,HP:6000918,HP:6001025,HP:6001034,HP:6001302,HP:6001426,HP:6001438
23,ABCB1 (OMIM:612244),0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
28,ABCB4 (OMIM:614972),0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
70,ACKR3 (OMIM:619215),0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
78,ACR (OMIM:620500),0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
102,ACTL7A (OMIM:620499),0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6122,ZNF644 (OMIM:614167),0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6130,ZP1 (OMIM:615774),0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6131,ZP2 (OMIM:618353),0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6132,ZP3 (OMIM:617712),0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
from data import prepare_data
x, features, labels = prepare_data(profils_omim, with_labels=True, normalize=False, n_pca=0)

In [18]:
import numpy as np
unique_rows = np.unique(features, axis=0)
print(f"Lignes totales : {features.shape[0]}")
print(f"Lignes uniques : {unique_rows.shape[0]}")
print(f"Doublons : {features.shape[0] - unique_rows.shape[0]}")

Lignes totales : 6139
Lignes uniques : 5957
Doublons : 182


In [116]:
checkpoint = torch.load('models/omim_S1.0_G3.0_K15_LR0.07_D10.pt', map_location='cpu', weights_only=False)

In [117]:
data = checkpoint["x"]
embeddings = checkpoint["embeddings"]
D_high = checkpoint["dist_kNNG"]
hyperparams = checkpoint["hyperparams"]

In [107]:
print("Nb inf :", np.isinf(D_high).sum())

Nb inf : 0


In [26]:
def get_ranking(distance_matrix):

    # According to this definition, reflexive ranks are set
    # to zero and non-reflexive ranks belong to {1,.., N − 1}.
    n = len(distance_matrix)
    sidx = np.argsort(distance_matrix, axis=1)
    Rank = np.zeros([n, n])
    ranks = np.tile(np.arange(n), (n, 1))
    np.put_along_axis(Rank, sidx[:, 1:], ranks[:, 1:], axis=1)
    return Rank

def get_coRanking(Rank_high, Rank_low):
    """
    Retourne une matrice binaire où une entrée vaut 1 si les deux noeuds sont "voisins" dans les deux matrices.
    """
    N = len(Rank_high)
    k = Rank_high.astype(int).ravel()
    l = Rank_low.astype(int).ravel()
    mask = (k > 0) & (l > 0)
    coRank = np.zeros([N-1, N-1])
    coRank = np.bincount(
        (k[mask] - 1) * (N - 1) + (l[mask] - 1),
        minlength=(N - 1) ** 2
    ).reshape(N - 1, N - 1)

    return coRank


In [27]:
def get_score(Rank_high, Rank_low):

    coRank = get_coRanking(Rank_high, Rank_low)
    N = len(coRank)+1

    df_score = pd.DataFrame(columns=['Qnx', 'Bnx'])
    Qnx = 0
    Bnx = 0
    for K in range(1, N):
        Qnx += sum(coRank[:K, K-1]) + sum(coRank[K-1, :K]) - coRank[K-1, K-1]
        Bnx += sum(coRank[:K, K-1]) - sum(coRank[K-1, :K])
        df_score.loc[len(df_score)] = [Qnx /(K*N), Bnx/(K*N)]
    
    return df_score

'''
    # Sommes cumulatives sur les lignes et colonnes
    row_cumsum = np.cumsum(coRank, axis=0)  # somme cumulée sur les lignes
    col_cumsum = np.cumsum(coRank, axis=1)  # somme cumulée sur les colonnes
    diag = np.diag(coRank)

    # Pour chaque K : sum(coRank[:K, K-1]) + sum(coRank[K-1, :K]) - coRank[K-1,K-1]
    # = col_cumsum[K-1, K-1] + row_cumsum[K-1, K-1] - diag[K-1]
    Ks = np.arange(1, N)
    qnx_increments = col_cumsum[Ks-1, Ks-1] + row_cumsum[Ks-1, Ks-1] - diag
    bnx_increments = col_cumsum[Ks-1, Ks-1] - row_cumsum[Ks-1, Ks-1]

    Qnx = np.cumsum(qnx_increments) / (Ks * N)
    Bnx = np.cumsum(bnx_increments) / (Ks * N)

    df_score = pd.DataFrame({'Qnx': Qnx, 'Bnx': Bnx})
    return df_score
''' 


"\n    # Sommes cumulatives sur les lignes et colonnes\n    row_cumsum = np.cumsum(coRank, axis=0)  # somme cumulée sur les lignes\n    col_cumsum = np.cumsum(coRank, axis=1)  # somme cumulée sur les colonnes\n    diag = np.diag(coRank)\n\n    # Pour chaque K : sum(coRank[:K, K-1]) + sum(coRank[K-1, :K]) - coRank[K-1,K-1]\n    # = col_cumsum[K-1, K-1] + row_cumsum[K-1, K-1] - diag[K-1]\n    Ks = np.arange(1, N)\n    qnx_increments = col_cumsum[Ks-1, Ks-1] + row_cumsum[Ks-1, Ks-1] - diag\n    bnx_increments = col_cumsum[Ks-1, Ks-1] - row_cumsum[Ks-1, Ks-1]\n\n    Qnx = np.cumsum(qnx_increments) / (Ks * N)\n    Bnx = np.cumsum(bnx_increments) / (Ks * N)\n\n    df_score = pd.DataFrame({'Qnx': Qnx, 'Bnx': Bnx})\n    return df_score\n"

In [28]:
def get_scalars(Qnx):
    
    N = len(Qnx) # total length of Qnx is smaller than number of samples
    K_max = 0
    val_max = Qnx[0] - 1/N
    for k in range(1, N):
        if val_max < (Qnx[k] - (k+1)/N):
            val_max = Qnx[k] - (k+1)/N
            K_max = k

    Qlocal = np.mean(Qnx[:K_max+1])
    Qglobal = np.mean(Qnx[K_max:])

    return Qlocal, Qglobal, K_max

In [29]:
def poincare_distance(x, eps=1e-5):
    boundary = 1 - eps
    nx = x.size(0)
    x = x.contiguous().view(nx, -1)

    norm_x = torch.sum(x ** 2, 1, keepdim=True)
    
    dot = x @ x.t()
    sqdist = norm_x + norm_x.t() - 2 * dot
    sqdist = sqdist.clamp(min=0) * 2 
    
    squnorm = 1 - torch.clamp(norm_x, 0, boundary)
    x = (sqdist / torch.mm(squnorm, squnorm.t())) + 1
    z = torch.sqrt(torch.pow(x, 2) - 1)
    
    return torch.log(x + z)

In [30]:
def average_precision_score_batched(D_high, D_low, k=10, batch_size=128):
    n = D_high.shape[0]
    ap_scores = []
    
    knn_high = np.argsort(D_high, axis=1)[:, 1:k+1]  # exclut la diagonale
    y_true = np.zeros((n, n), dtype=np.float32)
    np.put_along_axis(y_true, knn_high, 1.0, axis=1)

    for i in range(0, n, batch_size):
        y_batch = y_true[i:i+batch_size]
        s_batch = -D_low[i:i+batch_size]
        
        sorted_idx = np.argsort(-s_batch, axis=1)
        sorted_true = np.take_along_axis(y_batch, sorted_idx, axis=1)
        
        cumsum = np.cumsum(sorted_true, axis=1)
        k = np.arange(1, n + 1)
        precision_at_k = cumsum / k
        
        n_relevant = sorted_true.sum(axis=1)
        ap = (precision_at_k * sorted_true).sum(axis=1) / np.maximum(n_relevant, 1)
        ap_scores.extend(ap.tolist())
    
    return np.mean(ap_scores)

In [79]:
def get_quality_metrics(coord_high, coord_low, D_high, distance='euclidean', 
#k_neighbours=20, 
verbose=False):
    """
    Implementation of 'Scale-independent quality criteria' from Lee et al.    
    Parameters
    ----------
    coord_high : np.array
        Feature matrix of the sample in the high dimensional space.
    coord_low : np.array
        Low dimensional embedding of the sample.
    distance : str (default: 'euclidean')
        Distance metric to compute distanced between points in low dimendional 
        space. Possible parameters: 'euclidean' or 'poincare'.
    verbose: bool (default: False)
        A flag if to print the results of the computations.    
    k_neighbours: int (default: 20)
        k-nearest neighbours for setting
    Returns
    -------
    Qlocal: float
        Quality criteria for local qualities of the embedding.
        Range from 0 (bad) to 1 (good).
    Qglobal: float
        Quality criteria for global qualities of the embedding.
        Range from 0 (bad) to 1 (good).
    Kmax: int
        Kmax defines the split of the QNX curv.
    """

    Rank_high = get_ranking(D_high)

    if distance == 'euclidean':     
        D_low = pairwise_distances(coord_low)
    elif distance == 'poincare':
        D_low = poincare_distance(torch.DoubleTensor(coord_low)).numpy()
    else:
        raise NotImplementedError
    print(D_low.shape)
    print("Nb inf :", np.isinf(D_low).sum())
    print("==== Get ranking ====")
    Rank_low = get_ranking(D_low)
    print("==== Get score ====")
    df_score = get_score(Rank_high, Rank_low)
    print("==== Average precision ====")
    ap_scores = average_precision_score_batched(D_high, D_low)
    print ("==== Get scalars ====")
    Qlocal, Qglobal, Kmax = get_scalars(df_score['Qnx'].values)
    if verbose:
        print(f"Qlocal = {Qlocal:.2f}, Qglobal = {Qglobal:.2f}, Kmax = {Kmax}, ap_scores = {ap_scores}")

    return Qlocal, Qglobal, Kmax, ap_scores

In [118]:
Qlocal, Qglobal, Kmax, ap_scores = get_quality_metrics(data, embeddings, D_high, distance='poincare', verbose=True)

(6139, 6139)
Nb inf : 0
==== Get ranking ====
==== Get score ====
==== Average precision ====
==== Get scalars ====
Qlocal = 0.67, Qglobal = 0.80, Kmax = 37, ap_scores = 0.595683411147807
